# Post-processing a MODFLOW 6 model

Load the groundwater flow (**GWF**) model built and run in [`flopy-intro-gwf-only-a`](flopy-intro-gwf-only-a.ipynb), read its simulated heads and flows, and map them. By the end you will be able to load an existing MODFLOW 6 simulation from disk, pull results out of the head and cell-by-cell files, and plot them in both map view and cross section.

**Run [`flopy-intro-gwf-only-a`](flopy-intro-gwf-only-a.ipynb) first.** This notebook reads the output files that notebook writes; it does not build or run anything itself.

Post-processing is worth separating from model building because it is what you do over and over. A model is built once and then run and interpreted many times, often by someone who did not build it - so loading a simulation from disk, rather than keeping it in memory, is the normal way to work.


#### Imports and workspace

Import FloPy and Matplotlib, then point `ws` at the workspace notebook A wrote to and set `name` to the same simulation name. `require_gwf_output` (from the paired `mf6_notebook_helpers` module) is used below to check the model has actually been run.


In [ ]:
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
from mf6_notebook_helpers import require_gwf_output

In [ ]:
# exercise: point at the workspace and simulation name used in notebook A
# your code here


#### Load the simulation

Load the simulation that is already on disk with `flopy.mf6.MFSimulation.load()`, passing `sim_name` and `sim_ws`. FloPy reads the MODFLOW 6 input files and rebuilds the same `sim` object notebook A created, so everything you could do with the model there you can do here.

Then get the flow model out of the simulation with `sim.get_model()`. A simulation can hold several models, so ask for this one by name.


In [ ]:
# exercise: load the simulation from disk and get the flow model out of it
# your code here


Loading a simulation only reads the *input* files. Check that the *output* files are there too - that is, that notebook A was actually run - before trying to read results from them. `require_gwf_output()` raises a clear error pointing back to notebook A if the head or budget file is missing or empty.


In [ ]:
require_gwf_output(gwf, ws, hint="run flopy-intro-gwf-only-a.ipynb first")

#### Read the simulated heads

Get the head file object with `gwf.output.head()`. That object does not hold the heads themselves - it is a reader for the binary file - so ask it for what you want: `.get_times()` lists the times that were saved, and `.get_data()` returns the heads as a numpy array of shape `(nlay, nrow, ncol)`.

This model has a single steady-state stress period, so there is only one time.


In [ ]:
# exercise: open the head file and list the times it holds
# your code here


In [ ]:
# exercise: read the heads, then report the array shape and the head range
# your code here


#### Read the cell-by-cell flows

Get the cell-by-cell (budget) file object with `gwf.output.budget()`. It holds several kinds of record, one per package plus the flows between cells. The `.headers` table lists what is in the file; selecting the `text` and `imeth` columns and dropping duplicates gives the unique record types.


In [ ]:
# exercise: open the budget file and list the record types it holds
# your code here


`DATA-SPDIS` is the **specific discharge** - the groundwater flux, with an x, y, and z component in every cell. It is what you need to draw flow arrows. Read it with `.get_data(text="DATA-SPDIS")`, which returns a list with one entry per saved time, so take the first one.


In [ ]:
# exercise: read the specific discharge for the first (and only) saved time
# your code here


#### Map the heads and flows

Plot the results with `flopy.plot.PlotMapView()`. Use `.plot_array()` for the heads, `.plot_vector()` for the flow arrows, `.plot_bc()` for the boundary conditions, and `.plot_grid()` for the grid. `layer=2` selects the bottom layer, the one the well pumps from.

The legend and color bar below the marked block are given to you - they are fiddly, and not the point of the exercise.


In [ ]:
# exercise: map the head, the flow arrows, and the RIV and WEL boundaries in layer 3
# your code here

# create data outside of plot limits for legend data
mm.ax.plot(
    -100, -100, marker="s", lw=0, ms=4, mfc="red", mec="black", mew=0.5, label="Well"
)
mm.ax.plot(
    -100,
    -100,
    marker="s",
    lw=0,
    ms=4,
    mfc="blue",
    mec="black",
    mew=0.5,
    label="River cell",
)

# plot legend
plt.legend()

# plot colorbar
cb = plt.colorbar(cbv, ax=mm.ax, shrink=0.5)
cb.set_label(label="Head, ft", weight="bold")

**What to look for.** The colors are the simulated head in the bottom layer, and the arrows show the direction of groundwater flow. Heads are highest near the river cells (blue) along the right edge, where water enters the aquifer, and are drawn down toward the pumping well (red). The flow arrows point from the river toward the well — the main flow path in this system.


#### Look at the heads in cross section

A map shows one layer at a time. A **cross section** cuts down through all three layers at once, which is how you see what the low-permeability middle layer is doing.

Use `flopy.plot.PlotCrossSection()` with `line={"row": 10}` to slice along row 11 (rows are zero-based), the row the well is in. The methods are the same ones you used in map view: `.plot_array()` for the heads, `.plot_grid()` for the grid. Add `.contour_array()` to draw head contours on top, which makes the vertical gradient easy to see, and `fig.colorbar()` to label the colors. The figure and its axis labels are already set up in the cell.


In [ ]:
# the figure and its axis labels are set up for you
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(8, 3), layout="constrained")
    ax.set_xlabel("Distance along row 11, ft")
    ax.set_ylabel("Elevation, ft")

    # exercise: draw the cross section along the well row and contour the heads
    # your code here


**What to look for.** The section runs west (left) to east (right) through the well, and the three layers behave very differently. The top layer is unconfined, so the colored area stops at the water table, which slopes down to the east; its head falls about 21 ft across the row, from 341 ft to 320 ft, because recharge lands on it and the river drains it. The bottom layer barely moves at all — about 2 ft across the whole 10,000 ft — with a slight low at the well.

The thin middle layer is why. Its `k` is 0.01 ft/day against 50 and 200 above and below, so it nearly decouples the two aquifers. Look at the vertical head difference and note that it **reverses**: on the west side the top layer sits about 7 ft above the bottom one, the difference falls to zero near 7,000 ft, and by the river the bottom layer is about 9 ft *above* the top. West of that crossover water is moving down through the confining unit; east of it, up. That is a recharge area draining to a discharge area, and it is invisible in the map view you drew above.


## Recap

In this notebook you:

- Loaded an existing MODFLOW 6 simulation from disk with `flopy.mf6.MFSimulation.load()` and pulled the flow model out of it with `sim.get_model()`.
- Checked the model had been run before trying to read its output.
- Read the simulated heads with `gwf.output.head()` and the specific discharge from the cell-by-cell file with `gwf.output.budget()`.
- Mapped the head, flow directions, and boundary conditions in the bottom layer with `flopy.plot.PlotMapView()`.
- Cut a cross section through the well with `flopy.plot.PlotCrossSection()` and saw that the thin low-permeability middle layer nearly decouples the two aquifers, with flow moving down through it in the west and up through it near the river.
